# 1.4 Ingeniería de variables para la línea base

Actividad **1.4**:

| grupo | variables | n |
|---|---|---|
| sensores | `{volt,rotate,pressure,vibration}_{mean,std}_{3h,24h}` | 16 |
| errores | `error{1..5}_count_24h`, `error_total_count_24h` | 6 |
| mantenimiento | `h_since_comp{1..4}` | 4 |
| estáticas | `model_model{1..4}` (one-hot), `age` | 5 |

- **Causalidad.** Todas las variables usan solo información con marca de tiempo ≤ *t*. Las ventanas son retrospectivas (*t* − w, *t*].
- **Errores.** Se cuentan por tipo (además del total) para que el modelo pueda distinguir componentes en la tarea multiclase.
- **Mantenimiento.** Solo reemplazos con `datetime ≤ t` (ver 1.2). Los de 2014 inicializan la edad de cada componente.

El cálculo está en `src/features.py`, y las tablas las genera `00_pipeline.ipynb`. Este notebook las lee y las resume.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data import load_split
from src.features import FEATURE_GROUPS, ALL_FEATURES

print({g: len(v) for g, v in FEATURE_GROUPS.items()}, "→", len(ALL_FEATURES), "variables")
for s in ("train", "val"):
    t = load_split(s)
    print(f"{s:5s} {t.shape}  positivos={t['y_bin'].mean():.3%}  {t['datetime'].min()} → {t['datetime'].max()}")

{'sensores': 16, 'errores': 6, 'mantenimiento': 4, 'estaticas': 5} → 31 variables
train (580200, 36)  positivos=2.024%  2015-01-01 06:00:00 → 2015-08-30 23:00:00
val   (144000, 36)  positivos=1.842%  2015-09-01 00:00:00 → 2015-10-30 23:00:00


In [2]:
# Faltantes: solo la std de la primera hora de cada máquina (una sola lectura)
tr = load_split("train")
na = tr[ALL_FEATURES].isna().sum()
na[na > 0]

volt_std_3h          100
rotate_std_3h        100
pressure_std_3h      100
vibration_std_3h     100
volt_std_24h         100
rotate_std_24h       100
pressure_std_24h     100
vibration_std_24h    100
dtype: int64

## Resumen

- 31 variables: 16 de sensores, 6 de errores, 4 de mantenimiento y 5 estáticas.
- Los únicos faltantes son las desviaciones estándar de la primera hora de cada máquina (100 filas en train). En 1.5 se imputan con la mediana de train.